# WhatsApp agent workflow playground

Run cells from top to bottom, editing the inputs as needed. Expensive or data-changing actions are controlled by flags and are off by default. Raw chat messages and retrieved text are hidden unless you opt in. This notebook does not fine-tune model weights.

## 0. Find the repository and set your inputs

This step also checks whether the expected `.env` variables are loaded. It reports only the API key's status, never its value.

In [ ]:
from pathlib import Path
import os
import sys
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'src').is_dir() and (p / 'scripts').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('Open this notebook from inside the repository.')
os.chdir(REPO_ROOT)
load_dotenv(REPO_ROOT / '.env')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXAMPLE_CHAT_FILE = REPO_ROOT / 'data' / 'example_chat.txt'
PRIVATE_CHAT_FILES = [path for path in (REPO_ROOT / 'data').glob('*.txt') if path != EXAMPLE_CHAT_FILE]
CHAT_FILE = PRIVATE_CHAT_FILES[0] if PRIVATE_CHAT_FILES else EXAMPLE_CHAT_FILE  # Or set this to another export
CHROMA_PATH = REPO_ROOT / 'chroma_db'
from src.config import (
    SESSION_GAP_MINUTES, VECTOR_SEARCH_K, BM25_SEARCH_K,
    MAX_RANDOM_CHECKS, RANDOM_SESSIONS_PER_CHECK,
)
GAP_MINUTES = SESSION_GAP_MINUTES
SHOW_PRIVATE_TEXT = False  # Set True only when you want message text shown in notebook output
BUILD_INDEX = False        # Set True to call the embedding API and write/update Chroma
RUN_RETRIEVAL = False    # Enable after building/loading a Chroma index
RUN_TOOL_CHECK = False     # Enable after building/loading the default ./chroma_db
RUN_AGENT = False         # Set True to call the chat model
RUN_EVALUATION = False     # Set True to run evaluation cases (model/API calls)
RUN_UNIT_TESTS = False     # Set True to run tests/test_data_loader.py
print('Repository:', REPO_ROOT)
print('Python:', sys.executable)
print('Chat export:', CHAT_FILE.name if CHAT_FILE.exists() else 'not found')
print('Chroma directory exists:', CHROMA_PATH.exists())
ENV_VARIABLES = (
    'OPENAI_API_KEY', 'IMPERSONATED_NAME', 'MODEL_NAME', 'EMBEDDING_MODEL',
    'TEMPERATURE', 'MAX_TOKENS', 'CHUNK_SIZE', 'CHUNK_OVERLAP',
    'SESSION_GAP_MINUTES', 'VECTOR_SEARCH_K', 'BM25_SEARCH_K',
    'MAX_RANDOM_CHECKS', 'RANDOM_SESSIONS_PER_CHECK',
)
print('Environment variables:')
for variable in ENV_VARIABLES:
    status = 'loaded' if os.getenv(variable) else 'missing'
    print(f'  {variable}: {status}')


## 1. Configuration

Checks that the API key is available without displaying it. The `.env` file should stay private.

In [ ]:
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')
api_key_configured = bool(os.getenv('OPENAI_API_KEY'))
print('API key configured:', api_key_configured)
if api_key_configured:
    from src.config import (
        MODEL_NAME, EMBEDDING_MODEL, TEMPERATURE, MAX_TOKENS, CHUNK_SIZE,
        CHUNK_OVERLAP, SESSION_GAP_MINUTES, VECTOR_SEARCH_K, BM25_SEARCH_K,
        MAX_RANDOM_CHECKS, RANDOM_SESSIONS_PER_CHECK,
    )
    print('Chat model:', MODEL_NAME)
    print('Embedding model:', EMBEDDING_MODEL)
    print('Temperature / max tokens:', TEMPERATURE, '/', MAX_TOKENS)
    print('Chunk size / overlap:', CHUNK_SIZE, '/', CHUNK_OVERLAP)
    print('Session gap:', SESSION_GAP_MINUTES)
    print('Vector / BM25 top-k:', VECTOR_SEARCH_K, '/', BM25_SEARCH_K)
    print('Random checks / sessions per check:', MAX_RANDOM_CHECKS, '/', RANDOM_SESSIONS_PER_CHECK)
else:
    print('Skipped API-dependent settings; parsing and local tests can still run.')

## 2. Inspect and parse the export

This uses the same inspection function as `scripts/inspect_data.py`. The summary masks sender names. Parsed message text stays hidden unless `SHOW_PRIVATE_TEXT` is enabled.

In [ ]:
from collections import Counter
from src.data import parse_whatsapp_chat
from scripts.inspect_data import inspect

if not CHAT_FILE.exists():
    print('Set CHAT_FILE to an existing WhatsApp export first.')
else:
    inspect(CHAT_FILE, gap_minutes=GAP_MINUTES)
    messages = parse_whatsapp_chat(CHAT_FILE)
    print('First parsed message (text hidden):', {k: v for k, v in messages[0].items() if k != 'text'} if messages else 'No messages parsed.')
    if SHOW_PRIVATE_TEXT and messages:
        messages[:5]

## 3. Group messages into sessions

In [ ]:
from src.data import group_into_sessions

if 'messages' in globals() and messages:
    sessions = group_into_sessions(messages, gap_minutes=GAP_MINUTES)
    sizes = [len(session) for session in sessions]
    print(f'{len(sessions)} sessions; message counts per session: min={min(sizes)}, median={sorted(sizes)[len(sizes)//2]}, max={max(sizes)}')
    if SHOW_PRIVATE_TEXT:
        sessions[0]
else:
    print('Run the parse section first and make sure it found messages.')

## 4. Convert sessions to documents

These are the documents that will be embedded. This cell does not call the embedding API.

In [ ]:
from src.data import sessions_to_documents

if 'sessions' in globals() and sessions:
    documents = sessions_to_documents(sessions, source=CHAT_FILE.stem)
    print('Documents:', len(documents))
    print('First document metadata:', documents[0].metadata)
    if SHOW_PRIVATE_TEXT:
        print(documents[0].page_content)
else:
    print('Run the parse and session sections first.')

## 5. Build or update the Chroma index

This sends conversation text to the configured embedding API and writes a local database. Set `BUILD_INDEX = True` in section 0 only when you intend to do that.

In [ ]:
if BUILD_INDEX:
    if not CHAT_FILE.exists():
        raise FileNotFoundError(CHAT_FILE)
    from src.indexing.build_index import build_index
    vector_store = build_index(CHAT_FILE, CHROMA_PATH, gap_minutes=GAP_MINUTES)
    print('Indexed session documents:', vector_store._collection.count())
else:
    print('Index build skipped. Set BUILD_INDEX = True in section 0 to run it.')

bm25_json_path = CHROMA_PATH / 'bm25_documents.json'
if bm25_json_path.exists():
    import json
    with bm25_json_path.open(encoding='utf-8') as bm25_file:
        bm25_documents = json.load(bm25_file)
    print(f'BM25 JSON found: {bm25_json_path} ({len(bm25_documents)} documents)')
else:
    print(f'BM25 JSON not found: {bm25_json_path}')

## 6. Try retrieval with your own question

Edit `RETRIEVAL_QUESTION`. Search results can contain private chat text, so they are hidden unless `SHOW_PRIVATE_TEXT` is True.

In [ ]:
RETRIEVAL_QUESTION = 'what time are we meeting for the garden cleanup'  # Replace with your own test question
BM25_KEYWORDS = 'garden cleanup meeting'  # Replace with keywords for lexical search
RETRIEVAL_K = VECTOR_SEARCH_K

if RUN_RETRIEVAL:
    from src.indexing.vector_store import load_vector_store
    vector_store = load_vector_store(str(CHROMA_PATH))
    retrieved_docs = vector_store.similarity_search(RETRIEVAL_QUESTION, k=RETRIEVAL_K)
    print('Retrieved documents:', len(retrieved_docs))
    for i, doc in enumerate(retrieved_docs, 1):
        print(f'{i}. metadata={doc.metadata}')
        if SHOW_PRIVATE_TEXT:
            print(doc.page_content)
else:
    print('Retrieval skipped. Set RUN_RETRIEVAL = True in section 0 to query Chroma.')

## 7. Test the retrieval tools

This exercises `src/tools.py` itself. The tools module uses the default `./chroma_db` directory. It may show retrieved private text only when opted in above.

In [ ]:
if RUN_TOOL_CHECK:
    from src.tools import retrieve_relevant_sessions, retrieve_random_sessions, search_sessions_bm25
    tool_result = retrieve_relevant_sessions.invoke({'query': RETRIEVAL_QUESTION})
    print(tool_result if SHOW_PRIVATE_TEXT else f'Retrieval tool returned {len(tool_result)} characters; set SHOW_PRIVATE_TEXT=True to display.')
    bm25_result = search_sessions_bm25.invoke({'query': BM25_KEYWORDS})
    print(bm25_result if SHOW_PRIVATE_TEXT else f'BM25 tool returned {len(bm25_result)} characters for keywords: {BM25_KEYWORDS}')
    random_result = retrieve_random_sessions.invoke({'number': 1})
    print(random_result if SHOW_PRIVATE_TEXT else f'Random-session tool returned {len(random_result)} characters.')
else:
    print('Tool check skipped. Set RUN_TOOL_CHECK = True in section 0.')

## 8. Inspect the system prompt and metric

In [ ]:
try:
    from src.prompts import SYSTEM_PROMPT
except ModuleNotFoundError as error:
    if error.name != 'src.prompts':
        raise
    from src.prompts_example import SYSTEM_PROMPT
from src.evaluation.metrics import keyword_coverage

print('System prompt loaded:', bool(SYSTEM_PROMPT))
print('Synthetic metric check:', keyword_coverage('I usually order tea.', ['tea', 'coffee']))

## 9. Ask the agent one question

This calls the chat model and may call retrieval tools. Set `RUN_AGENT = True` to run it. The graph's tools currently use the default `./chroma_db`.

In [ ]:
AGENT_QUESTION = 'wie is silke'  # Replace with your question

if RUN_AGENT:
    from langchain_core.messages import HumanMessage
    from src.agent import app
    result = app.invoke(
        {'messages': [HumanMessage(content=AGENT_QUESTION)]},
        {'configurable': {'thread_id': 'notebook-playground'}},
    )
    print(result['messages'][-1].content)
else:
    print('Agent call skipped. Set RUN_AGENT = True in section 0 to call the model.')

## 10. Inspect and run the evaluation cases

Add one JSON object per line to `evals/questions.jsonl`. Running evaluation calls the model for every case and saves generated answers and retrieved context under `evals/results/`.

In [ ]:
from src.evaluation.dataset import load_cases

LOCAL_CASES = REPO_ROOT / 'evals' / 'questions.jsonl'
EXAMPLE_CASES = REPO_ROOT / 'evals' / 'questions.example.jsonl'
cases_path = LOCAL_CASES if LOCAL_CASES.exists() else EXAMPLE_CASES
cases = load_cases(cases_path)
print('Evaluation cases:', len(cases))
for i, case in enumerate(cases, 1):
    print(i, case['question'], '| expected answer supplied:', bool(case['expected_answer']), '| keywords:', len(case['expected_keywords']))

if RUN_EVALUATION:
    from src.evaluation.run_eval import run
    run(cases_path, REPO_ROOT / 'evals' / 'results' / 'notebook.jsonl')
else:
    print('Evaluation skipped. Set RUN_EVALUATION = True in section 0 to call the model for all cases.')

## 11. Run the parser and session unit tests

In [ ]:
import subprocess

if RUN_UNIT_TESTS:
    completed = subprocess.run(
        [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
        cwd=REPO_ROOT, text=True, capture_output=True
    )
    print(completed.stdout)
    print(completed.stderr)
    print('Test process exit code:', completed.returncode)
else:
    print('Tests skipped. Set RUN_UNIT_TESTS = True in section 0.')

## 12. Compatibility modules

These old import paths are wrappers. This checks that they point to the new implementations without rebuilding the index.

In [ ]:
from src.data_loader import parse_whatsapp_chat as legacy_parser
from src.data.whatsapp_parser import parse_whatsapp_chat as current_parser
print('Parser wrapper points to current function:', legacy_parser is current_parser)
if api_key_configured:
    from src.vector_store import load_vector_store as legacy_loader
    from src.indexing.vector_store import load_vector_store as current_loader
    print('Vector-store wrapper points to current function:', legacy_loader is current_loader)
else:
    print('Skipped vector-store wrapper check because it loads API configuration.')